In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Configure dimensions
batch_size = 4
seq_len = 4096  # Long context length where standard attention would OOM
num_heads = 16
head_dim = 64

# FlashAttention requires FP16 or BF16 precision and CUDA execution
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16

# Generate mock Q, K, V tensors with shape: (Batch, Heads, Seq_Len, Head_Dim)
Q = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=dtype)
K = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=dtype)
V = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=dtype)

print("--- Running Native PyTorch Scaled Dot-Product Attention ---")

# Enforce FlashAttention backend execution explicitly
with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
    # SDPA automatically executes fused CUDA tiling and online softmax under the hood
    output = F.scaled_dot_product_attention(
        query=Q, 
        key=K, 
        value=V, 
        attn_mask=None, 
        dropout_p=0.0, 
        is_causal=True # Automatically applies causal lower-triangular tiling
    )

print(f"Execution Successful!")
print(f"Output Tensor Memory Location: {output.device}")
print(f"Output Tensor Shape:           {output.shape} (Batch, Heads, Seq_Len, Head_Dim)")

--- Running Native PyTorch Scaled Dot-Product Attention ---


/usr/local/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Execution Successful!
Output Tensor Memory Location: cpu
Output Tensor Shape:           torch.Size([4, 16, 4096, 64]) (Batch, Heads, Seq_Len, Head_Dim)
